# FairRoute scaling workloads

This notebook is the reproducible entry point for the W3 FLOOD proxy grid. It writes no measured claims: all cards are computed from generated JSONL files.

In [ ]:
from pathlib import Path
import csv, json, hashlib, statistics
from workloads.datasets.flood_proxy import build_flood

ROOT = Path.cwd()
OUT = ROOT / 'workloads/fairroute/scaling/w3'


In [ ]:
cells = sorted({(k, r) for k, r in [(4, 1.0), (8, 1.0), (16, 1.0), (32, 1.0), (64, 1.0), (16, .5), (16, .8), (16, 1.25), (16, 1.5), (4, .5), (8, .8), (32, 1.25), (64, 1.5)]})
def card(rows, k, rho, seed):
    counts = {}
    for row in rows: counts[row['client_id']] = counts.get(row['client_id'], 0) + 1
    arrivals = [row['arrival_time_ns'] for row in rows]
    return {'workload': 'W3', 'proxy': 'PROXY', 'K': k, 'rho': rho, 'seed': seed,
            'requests': len(rows), 'clients': len(counts),
            'duration_ns': max(arrivals) - min(arrivals),
            'realised_rate_rps': len(rows) / max(1e-9, (max(arrivals) - min(arrivals)) / 1e9),
            'min_requests_per_client': min(counts.values()),
            'median_requests_per_client': statistics.median(counts.values()),
            'max_requests_per_client': max(counts.values()),
            'sha256': hashlib.sha256(json.dumps(rows, sort_keys=True).encode()).hexdigest()}

cards = []
for seed in (7, 100):
    for k, rho in cells:
        path = OUT / f'K{k}_L{rho}_s{seed}' / 'trace.jsonl'
        build_flood(rho, seed, 10000, path, K=k)
        rows = [json.loads(line) for line in path.read_text().splitlines()]
        card_path = path.with_name('card.json')
        card_path.write_text(json.dumps(card(rows, k, rho, seed), indent=2) + '\n')
        cards.append(card(rows, k, rho, seed))
validation = ROOT / 'validation'; validation.mkdir(exist_ok=True)
with (validation / 'workload_cards.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=cards[0]); writer.writeheader(); writer.writerows(cards)
print(f'generated {len(cards)} cards')
